<a href="https://colab.research.google.com/github/mona0101/Final-year-project-2026/blob/main/Making_the_Most_of_your_Colab_Subscription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install augly
! pip install librosa
# Mount drive if not already mounted
import google.colab.drive as drive
import os
if not os.path.exists('/content/drive/MyDrive/Colab Notebooks/'):
    drive.mount('/content/drive')
import re
import librosa
import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
import sys

# Fix for augly/torchaudio compatibility in Python 3.12 / Torchaudio 2.x
import torchaudio

# Define a mock class to replace the missing sox_effects module
class SoxPatch:
    @staticmethod
    def apply_effects_tensor(tensor, sample_rate, effects, channels_first=True):
        # Most augly transforms just need the tensor returned if sox isn't handling it
        return tensor, sample_rate
# Inject the patch into sys.modules so augly can find it
if not hasattr(torchaudio, 'sox_effects'):
    mock_sox = SoxPatch()
    sys.modules['torchaudio.sox_effects'] = mock_sox
    torchaudio.sox_effects = mock_sox

Mounted at /content/drive


In [6]:
!git clone https://github.com/mona0101/Final-year-project-2026.git
%cd Final-year-project-2026

Cloning into 'Final-year-project-2026'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 63 (delta 31), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 625.06 KiB | 5.79 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/Final-year-project-2026


In [7]:
from dataset_loader import DroneFusionDataset, get_loader
from augmentation import get_audio_transform, get_img_transform, get_rf_transform

class Args:
    dataset_dir = "/content/drive/MyDrive/Colab Notebooks/"
    batch_size = 4
    crop_size = 112
    #scale_size = 640

args = Args()
# Added augment=True to get_rf_transform to support the progression visualization
train_transform = {'audio': get_audio_transform(args, is_training=True, feature_type='mfcc'), 'video': get_img_transform(args, is_training=True, augment=True), 'rf': get_rf_transform(args, is_training=True, augment=True)}
test_transform = {'audio': get_audio_transform(args, is_training=False,feature_type='mfcc'), 'video': get_img_transform(args, is_training=False,  augment=False), 'rf': get_rf_transform(args, is_training=False,  augment=False)}
val_transform = {'audio': get_audio_transform(args, is_training=False,feature_type='mfcc'), 'video': get_img_transform(args, is_training=False,  augment=False), 'rf': get_rf_transform(args, is_training=False,  augment=True)}

audio_root = os.path.join(args.dataset_dir, 'Audio')
video_root = os.path.join(args.dataset_dir, 'Video')
rf_root = os.path.join(args.dataset_dir, 'RF_Spectrograms')

train_dataset = DroneFusionDataset(audio_root + '/Train', video_root + '/Train', rf_root + '/Train', 'Train', train_transform ,audio_feature_type='mfcc',audio_sr=44100) #mfcc= 44100, logmel=16000
test_dataset  = DroneFusionDataset(audio_root + '/Test',  video_root + '/Test',  rf_root + '/Test',  'Test', test_transform,  audio_feature_type='mfcc' )
val_dataset =DroneFusionDataset(audio_root + '/Validation',  video_root + '/Validation',  rf_root + '/Validation',  'Validation', val_transform,  audio_feature_type='mfcc' )

train_loader = get_loader(train_dataset, batch_size=args.batch_size, is_train=True)
test_loader  = get_loader(test_dataset,  batch_size=args.batch_size, is_train=False)
val_loader = get_loader(val_dataset, batch_size=args.batch_size, is_train=False)

for audio, video, rf, label in train_loader:
    print("Train batch:")
    print(audio.shape, video.shape, rf.shape, label.shape)
    break


Train batch:
torch.Size([4, 1, 40, 40]) torch.Size([4, 7, 3, 112, 112]) torch.Size([4, 1, 3, 112, 112]) torch.Size([4])


In [8]:
import torch
import os
from dataset_loader2 import DroneFusionDataset, get_loader
from augmentation import get_rf_transform

# 1. الإعدادات الأساسية
class Args:
    dataset_dir = '/content/drive/MyDrive/Colab Notebooks/'
    crop_size = 112
    batch_size = 32
    num_workers = 0

args = Args()

# 2. RF Transform فقط
train_transform = {
    'rf': get_rf_transform(args, is_training=True, augment=True)
}

val_transform = {
    'rf': get_rf_transform(args, is_training=False, augment=False)
}

# 3. اختيار RF فقط
current_modalities = ['rf']

# 4. المسارات
audio_root = os.path.join(args.dataset_dir, 'Audio')
video_root = os.path.join(args.dataset_dir, 'Video')
rf_root = os.path.join(args.dataset_dir, 'RF_Spectrograms')

# 5. إنشاء الـ Datasets
train_dataset = DroneFusionDataset(
    audio_root + '/Train',
    video_root + '/Train',
    rf_root + '/Train',
    'Train',
    train_transform,
    modalities=current_modalities
)

val_dataset = DroneFusionDataset(
    audio_root + '/Validation',
    video_root + '/Validation',
    rf_root + '/Validation',
    'Validation',
    val_transform,
    modalities=current_modalities
)

# 6. Loaders
train_loader = get_loader(train_dataset, batch_size=args.batch_size, is_train=True)
val_loader   = get_loader(val_dataset,   batch_size=args.batch_size, is_train=False)

# 7. اختبار القراءة
for audio, video, rf, labels in train_loader:
    print("RF batch loaded successfully:")
    print("Audio:", audio.shape)  # سيكون فارغ
    print("Video:", video.shape)  # سيكون فارغ
    print("RF:", rf.shape)        # هذا المهم
    print("Labels:", labels.shape)
    break

RF batch loaded successfully:
Audio: torch.Size([32, 0])
Video: torch.Size([32, 0])
RF: torch.Size([32, 1, 3, 112, 112])
Labels: torch.Size([32])


In [9]:
import os

print("Audio root contents:", os.listdir(audio_root))
print("Video root contents:", os.listdir(video_root))
print("RF root contents:", os.listdir(rf_root))

Audio root contents: ['Train', 'Test', 'Validation']
Video root contents: ['Validation', 'Test', 'Train']
RF root contents: ['Test', 'Train', 'Validation']


In [10]:
batch = next(iter(train_loader))
audio, video, rf, y = batch

print("audio:", type(audio), getattr(audio, "shape", None))
print("video:", video.shape)  # expected: (B, T, C, H, W)
print("rf:", rf.shape)
print("y:", len(y), y[:5])

audio: <class 'torch.Tensor'> torch.Size([32, 0])
video: torch.Size([32, 0])
rf: torch.Size([32, 1, 3, 112, 112])
y: 32 tensor([1, 0, 0, 0, 1])


In [11]:
import torch
import os
from dataset_loader2 import DroneFusionDataset, get_loader
from augmentation import get_rf_transform

class Args:
    dataset_dir = '/content/drive/MyDrive/Colab Notebooks/'
    crop_size = 112
    batch_size = 32
    num_workers = 0

args = Args()

train_transform = {
    'rf': get_rf_transform(args, is_training=True, augment=True)
}

val_transform = {
    'rf': get_rf_transform(args, is_training=False, augment=False)
}

current_modalities = ['rf']

audio_root = os.path.join(args.dataset_dir, 'Audio')
video_root = os.path.join(args.dataset_dir, 'Video')
rf_root = os.path.join(args.dataset_dir, 'RF_Spectrograms')

train_dataset = DroneFusionDataset(
    audio_root + '/Train',
    video_root + '/Train',
    rf_root + '/Train',
    'Train',
    train_transform,
    modalities=current_modalities
)

val_dataset = DroneFusionDataset(
    audio_root + '/Validation',
    video_root + '/Validation',
    rf_root + '/Validation',
    'Validation',
    val_transform,
    modalities=current_modalities
)

print("Train size:", len(train_dataset))

Train size: 8480


In [12]:
train_loader = get_loader(train_dataset, batch_size=args.batch_size, is_train=True)
val_loader   = get_loader(val_dataset, batch_size=args.batch_size, is_train=False)

In [13]:
batch = next(iter(train_loader))
audio, video, rf, labels = batch

print("RF:", rf.shape)
print("Labels:", labels.shape)

RF: torch.Size([32, 1, 3, 112, 112])
Labels: torch.Size([32])


In [14]:
torch.backends.cudnn.benchmark = True

In [15]:
def train_epoch(model, loader, criterion, optimizer):

    model.train()
    correct, total = 0, 0

    scaler = torch.amp.GradScaler("cuda")

    for audio, video, rf, y in loader:

        x = rf[:, 0]  # إزالة البعد الزمني (يصير 3,112,112)
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    return correct / total

In [16]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

torch.backends.cudnn.benchmark = True

Using device: cuda


In [17]:
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):

    train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_acc   = eval_epoch(model, val_loader)

    print(f"Epoch {epoch:02d} | Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

Epoch 01 | Train Acc 0.9108 | Val Acc 0.9664
Epoch 02 | Train Acc 0.9454 | Val Acc 0.9594
Epoch 03 | Train Acc 0.9480 | Val Acc 0.9758
